### Es 1
Hai a disposizione un file `data.csv` contenente dati mensili di passeggeri con due colonne:

- `date`: data in formato `YYYY-MM` (mese/anno)
- `passengers`: numero di passeggeri per quel mese


Costruisci un modello di **regressione polinomiale** che approssima l’andamento del numero di passeggeri nel tempo.

1. Carica il dataset.
2. Convertilo in un formato numerico utilizzando una colonna `mese_numerico` che conti i mesi a partire da gennaio 1949.
3. Applica una regressione polinomiale (grado a tua scelta).
4. Calcola l’RMSE tra i valori reali e quelli predetti.
5. Visualizza i dati reali e la curva stimata con Plotly.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import plotly.graph_objs as go

df = pd.read_csv("../data/data.csv")


df['mese_numerico'] = (
    (pd.to_datetime(df['date']) - pd.to_datetime('1949-01')).dt.days
)

df = df.dropna(subset=['passengers'])

grado = 3
X = df['mese_numerico'].values.reshape(-1, 1)
y = df['passengers'].values
poly = PolynomialFeatures(degree=grado)
X_poly = poly.fit_transform(X)
model = LinearRegression().fit(X_poly, y)
y_pred = model.predict(X_poly)

rmse = np.sqrt(mean_squared_error(y, y_pred))
print(f"RMSE: {rmse:.2f}")


fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=y, mode='markers', name='Dati reali'))
fig.add_trace(go.Scatter(x=df['date'], y=y_pred, mode='lines', name=f'Polinomio grado {grado}'))
fig.update_layout(title="Regressione polinomiale sui passeggeri",
                  xaxis_title="Data", yaxis_title="Numero passeggeri")
fig.show()

RMSE: 44.47


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[5], line 36, in aggiorna_grafico(grado=9)
     34 poly = PolynomialFeatures(degree=grado)
     35 X_poly = poly.fit_transform(x.reshape(-1, 1))
---> 36 model = LinearRegression().fit(X_poly, y)
        X_poly = array([[ 1.00000000e+00, -3.00000000e+00,  9.00000000e+00,
        -2.70000000e+01,  8.10000000e+01, -2.43000000e+02,
         7.29000000e+02, -2.18700000e+03,  6.56100000e+03,
        -1.96830000e+04],
       [ 1.00000000e+00, -2.93939394e+00,  8.64003673e+00,
        -2.53964716e+01,  7.46502347e+01, -2.19426447e+02,
         6.44980770e+02, -1.89585257e+03,  5.57265754e+03,
        -1.63802358e+04],
       [ 1.00000000e+00, -2.87878788e+00,  8.28741965e+00,
        -2.38577232e+01,  6.86813245e+01, -1.97718964e+02,
         5.69190958e+02, -1.63858003e+03,  4.71712433e+03,
        -1.35796003e+04],
       [ 1.00000000e

### Es2. 
Costruisci una web app con Dash che permette all’utente di scegliere il grado del polinomio per adattare un modello di regressione ai dati non lineari e vedere il risultato aggiornarsi dinamicamente.


1. Genera 100 punti x tra -3 e 3.

2. Calcola ad esempio y = x³ - x + rumore.

3. Costruisci un'interfaccia Dash con:
    - uno slider per scegliere il grado del polinomio (1–10),
    - un grafico Plotly che mostra i dati e la curva stimata.

4. Usa PolynomialFeatures + LinearRegression da scikit-learn per stimare la curva

In [5]:

import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.graph_objs as go
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

#np.random.seed(0)
x = np.linspace(-3, 3, 100)
y = x**3 - x + np.random.normal(0, 3, size=x.shape)

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H3("Regressione polinomiale interattiva"),
    dcc.Slider(
        id='grado-slider',
        min=1,
        max=10,
        step=1,
        value=3,
        marks={i: str(i) for i in range(1, 11)},
        tooltip={"placement": "bottom", "always_visible": True}
    ),
    dcc.Graph(id='poly-graph')
])

@app.callback(
    Output('poly-graph', 'figure'),
    Input('grado-slider', 'value')
)
def aggiorna_grafico(grado):
    poly = PolynomialFeatures(degree=grado)
    X_poly = poly.fit_transform(x.reshape(-1, 1))
    model = LinearRegression().fit(X_poly, y)
    y_pred = model.predict(X_poly)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers', name='Dati'))
    fig.add_trace(go.Scatter(x=x, y=y_pred, mode='lines', name=f'Polinomio grado {grado}'))
    fig.update_layout(
        title=f"Regressione polinomiale (grado {grado})",
        xaxis_title="x",
        yaxis_title="y"
    )
    return fig

if __name__ == '__main__':
    app.run(debug=True, port="8052")